In [10]:
import requests
import pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os


In [11]:
def configure():
    load_dotenv()

Collect daily/hourly data from 2022-01-01 to 2026-11-11

- The API can only send 5000 rows per request. A GET request includeds Net Generation, Demand, and Total interchange. Currently, the data we only require is Demand. A get request for the a single year is 34948 rows, including unseccary rows. It will take a at least 7 GET request to pull a full year of data. From Jan 2022 to Nov 2026, will require a minimum of 35 GET request. 

1) GET requests data
2) Transform dict to dataframe(df)
3) concat each df to main df

Raw Format: {"period":"2026-11-11T07","respondent":"DUK","respondent-name":"Duke Energy Carolinas","type":"D","type-name":"Demand","value-units":"megawatthours"}

In [ ]:
START_DATE = "2021-01-01T00"
END_DATE = "2026-11-07T00"

In [3]:
# split dates 2022-01-01T00 to 2026-11-07T00 into 34 sections
def getReq(start_date, end_date):
    # convert string to date format
    start = datetime.fromisoformat(start_date)
    end = datetime.fromisoformat(end_date)
    print(start)
    print(end)
    date_ranges = []

    while start < end:
        #next_date = min(start + timedelta(weeks=7), end)
        end_section = start + timedelta(weeks = 7)
        date_ranges.append((start.strftime("%Y-%m-%dT%H"), end_section.strftime("%Y-%m-%dT%H")))
        
        start = end_section + timedelta(days=1)
    return date_ranges


In [4]:
date_ranges = getReq(START_DATE, END_DATE)
df = pd.DataFrame()

2021-01-01 00:00:00
2026-11-07 00:00:00


In [ ]:
# iterate through length of req_dates where each index has two index. zero being the start date and one being the end date of the get request.
# iterate throught the list of dates to pull data from api. Once pulled concatinate to df. 
for date in date_ranges:
    print(f'Start date: {date[0]}')
    print(f'End date: {date[1]}')
    API_URL = f'https://api.eia.gov/v2/electricity/rto/region-data/data/?frequency=hourly&data[0]=value&facets[respondent][]=DUK&start={date[0]}&end={date[1]}&sort[0][column]=period&sort[0][direction]=desc&offset=0&length=5000{os.getenv('API_KEY')}'
    response = requests.get(API_URL)
    api_data = response.json()
    print("--------------------/n")
    print(response.status_code)
    # fetch https response 
    json_data = api_data["response"]["data"]
    df_holder = pd.DataFrame.from_dict(json_data)
    # remove all row that aren't a demand
    df_holder = df_holder.drop(df_holder[(df_holder.type != 'D')].index)
    # remove unessary columns
    df_holder = df_holder.drop(['respondent', 'respondent-name', 'type', 'type-name', 'value-units'], axis=1)
    df = pd.concat([df, df_holder], ignore_index=True)

Start date: 2021-01-01T00
End date: 2021-02-19T00
--------------------/n
200
Start date: 2021-02-20T00
End date: 2021-04-10T00


/var/folders/jy/xstwpqs1761_0t5mbjsw1fs00000gn/T/ipykernel_31914/3563258963.py:13: Pandas4Warning: Passing a list to DataFrame.from_dict is deprecated. Use the DataFrame constructor instead.
  df_holder = pd.DataFrame.from_dict(json_data)


--------------------/n
200
Start date: 2021-04-11T00
End date: 2021-05-30T00


/var/folders/jy/xstwpqs1761_0t5mbjsw1fs00000gn/T/ipykernel_31914/3563258963.py:13: Pandas4Warning: Passing a list to DataFrame.from_dict is deprecated. Use the DataFrame constructor instead.
  df_holder = pd.DataFrame.from_dict(json_data)


--------------------/n
200
Start date: 2021-05-31T00
End date: 2021-07-19T00


/var/folders/jy/xstwpqs1761_0t5mbjsw1fs00000gn/T/ipykernel_31914/3563258963.py:13: Pandas4Warning: Passing a list to DataFrame.from_dict is deprecated. Use the DataFrame constructor instead.
  df_holder = pd.DataFrame.from_dict(json_data)


... (76 outputs left out)

--------------------/n
200
Start date: 2026-10-02T00
End date: 2026-11-20T00


/var/folders/jy/xstwpqs1761_0t5mbjsw1fs00000gn/T/ipykernel_31914/3563258963.py:13: Pandas4Warning: Passing a list to DataFrame.from_dict is deprecated. Use the DataFrame constructor instead.
  df_holder = pd.DataFrame.from_dict(json_data)


--------------------/n
200


/var/folders/jy/xstwpqs1761_0t5mbjsw1fs00000gn/T/ipykernel_31914/3563258963.py:13: Pandas4Warning: Passing a list to DataFrame.from_dict is deprecated. Use the DataFrame constructor instead.
  df_holder = pd.DataFrame.from_dict(json_data)


In [ ]:
df.columns

In [7]:
df

,period,value
0,2021-02-19T00,15505
1,2021-02-18T23,15165
2,2021-02-18T22,14927
3,2021-02-18T21,14983
4,2021-02-18T20,15195
...,...,...
50550,2026-10-02T04,11117
50551,2026-10-02T03,12204
50552,2026-10-02T02,13331
50553,2026-10-02T01,14379


In [8]:
df.to_csv("DUK_hourly.csv", index=False)